# Emotion Scoring

## Imports

Load data tools and the Transformers pipeline API.

In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline

/Users/dylanhuang/micromamba/envs/df_ae2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device Selection

Use GPU if available for faster inference.

In [2]:
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

## Load Sentiment-Scored Tweets

Read the dataset produced in the sentiment step and order by ticker/date.

In [3]:
tweets_df = pd.read_parquet("../data/dataset/stock_tweets_sentiment_nomerge.parquet")

tweets_df = tweets_df.sort_values(["ticker", "created_at"]).reset_index(drop=True)


## Load Emotion Classifier

Initialize the pretrained emotion model for multi-class scoring.

In [4]:
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None, device=device)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1828.03it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Quick Sanity Check

Run the classifier on a sample sentence.

In [5]:
test = classifier("This is so exciting!")[0]

## Inspect Sample Output

Review raw model scores for each emotion.

In [6]:
test

[{'label': 'joy', 'score': 0.8689905405044556},
 {'label': 'surprise', 'score': 0.09550558775663376},
 {'label': 'neutral', 'score': 0.02346690557897091},
 {'label': 'anger', 'score': 0.005241710226982832},
 {'label': 'fear', 'score': 0.0029143420979380608},
 {'label': 'sadness', 'score': 0.0025007587391883135},
 {'label': 'disgust', 'score': 0.0013800475280731916}]

## Emotion Scoring Function

Return a score for each emotion label.

In [7]:
def emotion_analysis(text):
    analysis = classifier(text, top_k=None)
    return [e["score"] for e in analysis]

## Test the Function

Verify the helper on a simple example.

In [8]:
emotion_analysis("This is so exciting!")

[0.8689905405044556,
 0.09550558775663376,
 0.02346690557897091,
 0.005241710226982832,
 0.0029143420979380608,
 0.0025007587391883135,
 0.0013800475280731916]

## Score All Tweets

Expand emotion scores into columns and compute dataset-wide percentiles.

In [9]:
tweets_df[['emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy', 'emotion_neutral', 'emotion_sadness', 'emotion_surprize']] = tweets_df['text'].apply(emotion_analysis).apply(pd.Series)

emotion_cols = [
    'emotion_anger', 'emotion_disgust', 'emotion_fear',
    'emotion_joy', 'emotion_neutral', 'emotion_sadness', 'emotion_surprize'
]

# Percentile (0–1) across the whole dataset
for c in emotion_cols:
    tweets_df[c + '_pct'] = tweets_df[c].rank(pct=True)


## Preview Results

Inspect emotion feature columns.

In [10]:
tweets_df

,ticker,text,created_at,user_id,date,sentiment,emotion_anger,emotion_disgust,emotion_fear,emotion_joy,emotion_neutral,emotion_sadness,emotion_surprize,emotion_anger_pct,emotion_disgust_pct,emotion_fear_pct,emotion_joy_pct,emotion_neutral_pct,emotion_sadness_pct,emotion_surprize_pct
0,AAPL,summary of yesterdays webcast featuring wynn g...,2013-12-31 23:10:08+00:00,1864753100,2013-12-31,4,0.677257,0.148463,0.143604,0.021908,0.004223,0.003426,0.001120,0.605470,0.348441,0.761059,0.341360,0.083108,0.214632,0.197516
1,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 01:18:36+00:00,1937591882,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,0.004223,0.003426,0.001120,0.605470,0.348441,0.761059,0.341360,0.083108,0.214632,0.197516
2,AAPL,iphone users are more intelligent than samsung...,2014-01-01 01:52:31+00:00,23954327,2014-01-01,5,0.651749,0.290809,0.032565,0.017082,0.004068,0.001948,0.001778,0.562480,0.785138,0.180923,0.243422,0.074414,0.070304,0.430871
3,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:29:29+00:00,1933063572,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,0.004223,0.003426,0.001120,0.605470,0.348441,0.761059,0.341360,0.083108,0.214632,0.197516
4,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:59:03+00:00,1938270918,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,0.004223,0.003426,0.001120,0.605470,0.348441,0.761059,0.341360,0.083108,0.214632,0.197516
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106333,XOM,t active morning movers at t nyse t exxon mobil …,2015-12-28 17:15:13+00:00,2342763212,2015-12-28,5,0.793153,0.089309,0.077659,0.018933,0.011689,0.007353,0.001905,0.781047,0.177434,0.470923,0.286050,0.474177,0.514482,0.468055
106334,XOM,divest from stopcommoncore optout because chil...,2015-12-28 19:39:46+00:00,4399710563,2015-12-28,1,0.763934,0.115369,0.071567,0.021422,0.012145,0.009175,0.006386,0.737070,0.256536,0.439095,0.333601,0.488936,0.602386,0.859171
106335,XOM,zsl stock forum zsl gold uslv zsl investing na...,2015-12-29 16:52:36+00:00,2181314366,2015-12-29,5,0.488881,0.242657,0.201586,0.045692,0.012491,0.007334,0.001360,0.266922,0.625600,0.925934,0.646072,0.500263,0.513626,0.298238
106336,XOM,nptn recent news updated tuesday december pm g...,2015-12-29 19:03:17+00:00,2197054086,2015-12-29,1,0.456476,0.355527,0.112046,0.067945,0.003791,0.003206,0.001009,0.202848,0.928201,0.644962,0.791015,0.059649,0.193402,0.154526


## Save Output

Write the emotion-scored dataset to parquet for downstream use.

In [11]:
tweets_df.to_parquet('../data/dataset/stock_tweets_sentiment_emotion_nomerge.parquet',index=False)